# 01 手寫數字資料觀察與視覺化

本 notebook 先從資料本身出發，說明：一張灰階圖片在電腦中其實就是一個數字矩陣。`load_digits` 的像素值範圍是 0 到 16，不是常見影像的 0 到 255；若要縮放到 0 到 1，應除以 16。接著我們會把每張 8x8 圖片攤平成 64 維 feature vector，並用 PCA / t-SNE 投影到 2D 平面觀察分群。

> 重點：降維在此處只用來視覺化，不作為正式模型比較的訓練資料。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. 載入資料集

`load_digits` 是 scikit-learn 內建的手寫數字資料集，每張圖片是 8x8 灰階影像，類別是 0 到 9。這個資料集已經是低解析度、低灰階等級的資料，像素值範圍為 0 到 16。

In [ ]:
digits = load_digits()
X = digits.data          # shape: (n_samples, 64)
images = digits.images   # shape: (n_samples, 8, 8)
y = digits.target        # labels: 0-9

print('X shape:', X.shape)
print('images shape:', images.shape)
print('y shape:', y.shape)
print('class names:', digits.target_names)

## 2. 像素值範圍

`load_digits` 的像素值範圍是 0 到 16。這代表它不是原始 0 到 255 的一般灰階影像，而是已整理成 16 階亮度的 8x8 小圖。


In [ ]:
print('pixel min:', images.min())
print('pixel max:', images.max())
print('若要縮放到 0~1，可使用 images / 16.0 或 X / 16.0')


## 3. 看看手寫數字長什麼樣

先不要急著建模，先觀察圖片與標籤。

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for ax, image, label in zip(axes.ravel(), images[:20], y[:20]):
    ax.imshow(image, cmap='gray_r')
    ax.set_title(f'label={label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. 灰階圖片就是數字矩陣

下面印出同一張圖片的 8x8 array。數字越大，代表像素越亮。注意：此處最大值是 16，不是 255。

In [ ]:
idx = 0
print('label:', y[idx])
print('8x8 image array:')
print(images[idx])

plt.figure(figsize=(3, 3))
plt.imshow(images[idx], cmap='gray_r')
plt.title(f'label={y[idx]}')
plt.axis('off')
plt.show()

## 5. Flatten：把 8x8 圖片攤平成 64 維

傳統機器學習模型通常吃的是一列特徵向量。8x8 圖片可以攤平成 64 個數字。

In [ ]:
print('原本 images[0] shape:', images[0].shape)
print('攤平後 X[0] shape:', X[0].shape)
print('X[0] 前 20 個特徵:', X[0][:20])

## 6. 類別數量是否平均

分類資料如果某些類別很多、某些類別很少，模型可能會偏向多數類別。

In [ ]:
classes, counts = np.unique(y, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar(classes, counts)
plt.xticks(classes)
plt.xlabel('digit')
plt.ylabel('count')
plt.title('Class distribution')
plt.show()

for cls, count in zip(classes, counts):
    print(f'digit {cls}: {count} samples')

## 7. 每個數字的平均圖

把同一個類別的圖片平均起來，可以看出每個數字的大致形狀。

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for digit, ax in enumerate(axes):
    mean_img = images[y == digit].mean(axis=0)
    ax.imshow(mean_img, cmap='gray_r')
    ax.set_title(str(digit))
    ax.axis('off')
plt.suptitle('Average image per digit')
plt.tight_layout()
plt.show()

## 8. PCA：把 64 維資料投影到 2D

PCA 是線性降維方法。此處用來觀察資料是否有自然分群。

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=18, alpha=0.85)
plt.legend(*scatter.legend_elements(), title='Digits', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('Digits projected to 2D by PCA')
plt.tight_layout()
plt.show()

print('explained variance ratio:', pca.explained_variance_ratio_)
print('total explained variance:', pca.explained_variance_ratio_.sum())

## 9. t-SNE：用非線性降維觀察分群

`t-SNE` 常用於資料視覺化。它可以讓局部相近的資料在 2D 圖中靠近，但它不是正式模型評比的訓練資料。

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=RANDOM_STATE)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', s=18, alpha=0.85)
plt.legend(*scatter.legend_elements(), title='Digits', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('Digits projected to 2D by t-SNE')
plt.tight_layout()
plt.show()

## 課後練習與思考題

這一組練習的目標不是提高模型分數，而是建立資料直覺：影像在電腦中其實是一組數字，而不同數字類別在高維空間中可能有可視化的分群趨勢。

### 練習 1：觀察同一個 label 的筆跡差異

找出某一個指定數字的 5 張不同手寫圖片，觀察同一個 label 底下的筆跡差異。請特別留意筆畫粗細、傾斜角度與數字在格子中的位置。

In [ ]:
# 練習 1 參考答案：找出某一個數字的 5 張不同手寫圖片

target_digit = 3
sample_count = 5
sample_indices = np.where(y == target_digit)[0][:sample_count]

fig, axes = plt.subplots(1, sample_count, figsize=(10, 2))
for ax, idx in zip(axes, sample_indices):
    ax.imshow(images[idx], cmap='gray_r')
    ax.set_title(f'idx={idx}')
    ax.axis('off')
plt.suptitle(f'Examples of digit {target_digit}')
plt.tight_layout()
plt.show()



#### 練習 1 參考答案：觀察同一個 label 的筆跡差異

同樣都是相同 label，筆畫粗細、傾斜角度、數字在格子中的位置仍然不同。這也是影像分類不能只靠單一像素規則判斷的原因。

### 練習 2：找出最亮像素位置

選一張指定 label 的圖片，印出 `8x8 array`，並在圖上標出灰階值最大的像素位置，理解亮度與數字筆畫的關係。

In [ ]:
# 練習 2 參考答案：印出 8x8 array，並標出最亮的像素位置

target_digit = 3
sample_index = np.where(y == target_digit)[0][0]
sample_image = images[sample_index]

max_value = sample_image.max()
max_positions = np.argwhere(sample_image == max_value)

print('label:', y[sample_index])
print('sample index:', sample_index)
print('8x8 image array:')
print(sample_image)
print('max pixel value:', max_value)
print('max pixel positions [row, col]:')
print(max_positions)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(sample_image, cmap='gray_r')
ax.set_title(f'label={y[sample_index]}, max={max_value}')
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.grid(color='lightgray', linewidth=0.5)

for row, col in max_positions:
    ax.scatter(col, row, s=220, facecolors='none', edgecolors='red', linewidths=2)
    ax.text(col, row, str(int(max_value)), color='red', ha='center', va='center', fontsize=9, fontweight='bold')

plt.show()

#### 練習 2 參考答案：找出最亮像素位置

`load_digits` 的像素值範圍是 0 到 16，數值越大代表該位置越接近筆畫中心或筆畫較亮的位置。`max pixel positions [row, col]` 使用的是矩陣座標：第一個數字是列，也就是垂直方向；第二個數字是欄，也就是水平方向。

同一張圖片可能有多個像素同時達到最大值，這不代表模型只看這些像素就能分類，而是幫助學生把「圖片」和「數字矩陣」連在一起。

### 練習 3：用 PCA 觀察兩個可能混淆的數字

選兩個視覺上可能混淆的數字，例如 `3` 與 `8`，只取這兩類在 PCA 2D 平面中觀察是否有重疊。這裡的 PCA 只是視覺化觀察，不是正式分類模型。

In [ ]:
# 練習 3 參考答案：用 PCA 觀察兩個可能混淆的數字

# 可以改成 [4, 9]、[1, 7] 等組合觀察不同數字對。
target_pair = [3, 8]
pair_mask = np.isin(y, target_pair)

plt.figure(figsize=(7, 5))
for digit in target_pair:
    digit_mask = pair_mask & (y == digit)
    plt.scatter(
        X_pca[digit_mask, 0],
        X_pca[digit_mask, 1],
        s=22,
        alpha=0.75,
        label=f'digit {digit}'
    )

plt.title(f'PCA 2D view: digits {target_pair[0]} vs {target_pair[1]}')
plt.xlabel('PCA component 1')
plt.ylabel('PCA component 2')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

#### 練習 3 參考答案：用 PCA 觀察兩個可能混淆的數字

若兩個類別在 PCA 2D 平面中明顯重疊，代表只靠這兩個投影方向很難把它們完全分開。這個觀察可以幫助學生理解：人眼看到的是形狀，但模型看到的是 64 個像素特徵。

需要特別提醒：PCA 2D 圖只是把 64 維資料壓到 2 維後的觀察結果，不等於真正的分類模型，也不代表原始 64 維空間一定無法分開。下一個 notebook 會正式用 SVM 建立分類器，檢查模型在訓練資料與測試資料上的表現。